In [1]:
!pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html
!pip install torchtext    

In [3]:
import math
from typing import Tuple
import copy
import time
from datetime import datetime

import torch
from torch import nn, Tensor
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import dataset
from torchtext.datasets import WikiText2
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

[EPOCH]: 0, [TRAIN LOSS]: 3.161840, [VAL LOSS]: 3.115810
[EPOCH]: 50, [TRAIN LOSS]: 0.751510, [VAL LOSS]: 0.736730
[EPOCH]: 100, [TRAIN LOSS]: 0.132355, [VAL LOSS]: 0.133143
[EPOCH]: 150, [TRAIN LOSS]: 0.058727, [VAL LOSS]: 0.061338
[EPOCH]: 200, [TRAIN LOSS]: 0.050194, [VAL LOSS]: 0.052464
[EPOCH]: 250, [TRAIN LOSS]: 0.044302, [VAL LOSS]: 0.046269
[EPOCH]: 300, [TRAIN LOSS]: 0.038543, [VAL LOSS]: 0.040241
[EPOCH]: 350, [TRAIN LOSS]: 0.033061, [VAL LOSS]: 0.034506
[EPOCH]: 400, [TRAIN LOSS]: 0.027974, [VAL LOSS]: 0.029188
[EPOCH]: 450, [TRAIN LOSS]: 0.023358, [VAL LOSS]: 0.024364
[EPOCH]: 500, [TRAIN LOSS]: 0.019253, [VAL LOSS]: 0.020076
[EPOCH]: 550, [TRAIN LOSS]: 0.015667, [VAL LOSS]: 0.016332
[EPOCH]: 600, [TRAIN LOSS]: 0.012588, [VAL LOSS]: 0.013118
[EPOCH]: 650, [TRAIN LOSS]: 0.009987, [VAL LOSS]: 0.010405
[EPOCH]: 700, [TRAIN LOSS]: 0.007825, [VAL LOSS]: 0.008149
[EPOCH]: 750, [TRAIN LOSS]: 0.006053, [VAL LOSS]: 0.006302
[EPOCH]: 800, [TRAIN LOSS]: 0.004623, [VAL LOSS]: 0.004812


In [4]:
x =  torch.tensor([3100,-3000], dtype=torch.float).to(device)
model(x)

tensor([107.0707], grad_fn=<AddBackward0>)

In [ ]:
# now train multiplication

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")

N_train = 10000  # number of training samples
N_val = 1000
N_test = 1000
D = 2  # input dimension
d = 300 # inner hidden dimension
C = 1  # output dimension
val_lower = 1
val_upper = 1000
test_lower = 1000
test_upper = 10000

X_train = torch.rand(N_train, D).to(device)  # (N, D)
y_train = torch.prod(X_train, axis=-1).reshape(-1, C)  # (N, C)

X_val = torch.rand(N_val, D).to(device)  # (N, D)
y_val = torch.prod(X_val, axis=-1).reshape(-1, C)  # (N, C)

X_test = torch.rand(N_test, D).to(device)  # (N, D)
y_test = torch.prod(X_test, axis=-1).reshape(-1, C)  # (N, C)


lr = 1e-2  # Learning rate

model = torch.nn.Sequential(torch.nn.Linear(D, d), 
                            torch.nn.ReLU(),
                            torch.nn.Linear(d, d),
                            torch.nn.ReLU(),
                            torch.nn.Linear(d, d),
                            torch.nn.ReLU(),
                            torch.nn.Linear(d, d),
                            torch.nn.ReLU(),
                            torch.nn.Linear(d,C))  # model
model.to(device)

criterion = torch.nn.MSELoss()  # loss function
optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # optimizer

for epoch in range(1000):
    y_pred = model(X_train)  # forward step
    loss = criterion(y_pred, y_train)  # compute loss
    loss.backward()  # backprop (compute gradients)
    optimizer.step()  # update weights (gradient descent step)
    optimizer.zero_grad()  # reset gradients
    if epoch % 50 == 0:
        y_pred_val = model(X_val)
        val_loss = criterion(y_pred_val, y_val)
        print(f"[EPOCH]: {epoch}, [TRAIN LOSS]: {loss.item():.9f}, [VAL LOSS]: {val_loss.item():.9f}")
        
# now evaluate on test set
y_pred_test = model(X_test)
test_loss = criterion(y_pred_test, y_test)
accuracy = 100 - 100*torch.mean(abs(y_pred_test - y_test)/y_test).item()
print(f"FINAL TEST LOSS = {test_loss.item():.9f}, ACCURACY: {accuracy:.9f}%")
        